# D2: diagnosi dei siti rilevati, modello invariato

Questo notebook legge **`02_collapse.parquet` e `02_sites.parquet` della stessa run** e verifica
perché D2 può avere meno di dieci siti non ambigui. È autonomo: non richiede gli altri notebook,
un clone del repository, Chronos o una GPU. Runtime Colab **CPU** sufficiente.

**Ambito:** diagnosi della misura, della selezione e degli input del modello attuale.
Nessuna raccolta, MCMC, correzione del rilevatore o modifica delle soglie. La funzione
`model_D2_movement` è copiata integralmente dagli allegati, nei quali è identica.
I Parquet originali vengono soltanto letti; i risultati diagnostici vanno in una nuova directory.

Gli output precedenti di `new_analysis_v2` riportavano 67 siti stride e 1 patch, quest'ultimo
a 240 Hz per `p32-s16 / tsmixup`. Questi sono **riferimenti storici da verificare**, non risultati
precostituiti: tutti i conteggi qui sotto sono ricalcolati dai file selezionati.

**Uso:** eseguire dall'inizio. In Colab montare Drive quando richiesto. Il percorso predefinito è
`MyDrive/patchAliasing/full/d3_15model_v1/data`, la run di `new_analysis_v2`.
Per la run localisation impostare esplicitamente la sua cartella `data`, senza mescolare i file.
Alla fine leggere il riepilogo e i grafici, poi discutere eventuali modifiche a D2.


**Estensione diagnostica:** sezioni 12–14, sensibilità 0.60–0.95 e confronto continuo con controlli a ±4/±6 Hz. La soglia D2 resta 0.75.


**Audit 0.90:** sezioni 15–17, elenco completo dei siti, stabilità tra repliche e compatibilità di fondamentale e distanze con D2. Nessun refit.


## 1. Impostazioni e dipendenze

La diagnostica usa NumPy, Pandas, Matplotlib e un motore Parquet. In Colab vengono installate
solo le dipendenze mancanti; localmente viene indicato il comando necessario se mancano.
PyMC è richiesto soltanto per l'opzione `BUILD_MODEL_GRAPH`, che costruisce il modello senza campionare.


In [ ]:
from __future__ import annotations
import os, sys, json, hashlib, importlib.util, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
from dataclasses import dataclass

try:
    IS_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IS_COLAB = False

# Cambiare questa sola cartella per diagnosticare un'altra run.
SOURCE_DATA_DIR = os.environ.get('D2_SOURCE_DATA_DIR', '')
OUTPUT_ROOT = os.environ.get('D2_OUTPUT_ROOT', '')
BUILD_MODEL_GRAPH = False  # opzionale; nessun campionamento, nemmeno se True

required = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if not any(importlib.util.find_spec(engine) for engine in ('pyarrow', 'fastparquet')):
    missing.append('pyarrow')
if missing:
    if not IS_COLAB:
        raise RuntimeError('Dipendenze mancanti. Installare nel kernel: ' + ' '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
PARQUET_ENGINE = 'pyarrow' if importlib.util.find_spec('pyarrow') else 'fastparquet'
pd.set_option('display.max_rows', 35)
pd.set_option('display.max_columns', 15)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': .2, 'axes.spines.top': False, 'axes.spines.right': False})

FS, BAND = 512, (2.0, 250.0)
GENERATORS = ('tsmixup', 'kernelsynth')
EXPECTED_MODELS = [(8,8), (16,8), (16,12), (16,16), (24,8), (24,12), (24,16),
                   (24,20), (24,24), (32,8), (32,12), (32,16), (32,20), (32,24), (32,32)]

@dataclass
class Config:
    site_merge_tol_hz: float = 1.5
    site_assignment_tol_hz: float = 1.5
    site_ambiguity_hz: float = .25
    collapse_step: float = 1.0
    min_d2_sites: int = 10

CFG = Config()
ROPE_SLOPE = .1
DETECTOR = dict(rel_thr=.02, dip_ratio=.75, window_hz=6.0)
WARNINGS = []
TABLES = {}
FIGURES = []

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def warning(message):
    if message not in WARNINGS:
        WARNINGS.append(message)
        print('NOTA:', message)


## 2. Lettura dei Parquet e provenienza

Si usano esclusivamente i due file della cartella selezionata. Nessuna ricerca automatica in altre
run e nessun ripiego su dati smoke. Il manifest della raccolta è facoltativo: se presente se ne
controllano hash e configurazione. File privi di manifest restano diagnosticabili, con provenienza
non verificata. Una discrepanza negli hash interrompe la lettura.


In [ ]:
if not SOURCE_DATA_DIR:
    if IS_COLAB:
        SOURCE_DATA_DIR = '/content/drive/MyDrive/patchAliasing/full/d3_15model_v1/data'
    else:
        ancestors = [Path.cwd(), *Path.cwd().parents]
        repo = next((p for p in ancestors if (p / 'chronos/bayesian').is_dir()), Path.cwd())
        SOURCE_DATA_DIR = str(repo / 'chronos/bayesian/_run/full/d3_15model_v1/data')
if IS_COLAB and str(SOURCE_DATA_DIR).startswith('/content/drive/') and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path(SOURCE_DATA_DIR).expanduser().resolve()
INPUTS = {name: DATA_DIR / f'02_{name}.parquet' for name in ('collapse', 'sites')}
missing_files = [str(p) for p in INPUTS.values() if not p.is_file()]
if missing_files:
    raise FileNotFoundError('Parquet mancanti nella run selezionata: ' + ', '.join(missing_files)
                            + '. Impostare SOURCE_DATA_DIR sulla cartella che contiene entrambi.')
INPUT_HASHES = {name: sha256(path) for name, path in INPUTS.items()}
manifest_path = DATA_DIR / 'collection_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else {}
SOURCE_CONFIG = manifest.get('design', {}).get('config', {})
manifest_verified = bool(manifest)
for name, digest in INPUT_HASHES.items():
    entry = manifest.get('merged', {}).get(name, {})
    if entry.get('sha256') and entry['sha256'] != digest:
        raise ValueError(f'Hash del Parquet {name} diverso dal manifest. Non combinare revisioni.')
    if not entry.get('sha256'):
        manifest_verified = False
if not manifest_verified:
    warning('Provenienza non completamente verificata: manifest/hash mancanti; conservati gli hash attuali.')
if manifest and manifest.get('status') != 'complete':
    warning('Il manifest non dichiara una raccolta completa.')
if SOURCE_CONFIG.get('smoke') or manifest.get('design', {}).get('reportable') is False:
    warning('Sorgente SMOKE/non-reportable: risultati utili solo per il controllo software e del disegno.')
for key, expected in vars(CFG).items():
    if key in SOURCE_CONFIG and SOURCE_CONFIG[key] != expected:
        warning(f'Configurazione sorgente {key}={SOURCE_CONFIG[key]}; snapshot D2 attuale={expected}.')
if SOURCE_CONFIG.get('generators') and set(SOURCE_CONFIG['generators']) != set(GENERATORS):
    warning('Generatori della sorgente diversi dai due generatori del disegno completo.')

collapse = pd.read_parquet(INPUTS['collapse'], engine=PARQUET_ENGINE)
sites = pd.read_parquet(INPUTS['sites'], engine=PARQUET_ENGINE)
PROVENANCE = dict(source_data_dir=str(DATA_DIR), input_sha256=INPUT_HASHES,
                  manifest_sha256=sha256(manifest_path) if manifest else None,
                  manifest_verified=manifest_verified, source_config=SOURCE_CONFIG,
                  diagnostic_config=vars(CFG), detector=DETECTOR,
                  python=platform.python_version(), packages={
                      'numpy': np.__version__, 'pandas': pd.__version__,
                      'matplotlib': plt.matplotlib.__version__})
display(pd.DataFrame([dict(table=k, rows=len(v), file=str(INPUTS[k]), sha256=INPUT_HASHES[k])
                      for k, v in [('collapse', collapse), ('sites', sites)]]))


### 2.1 Schema, copertura e unità di osservazione

Una riga `sites` riassume **geometria × generatore × replica × ramo**. `rep=-1` indica la
curva media, non un'ulteriore replica indipendente. Il gate dei dieci siti usa solo queste curve
medie e i background generati; il modello usa le righe con `rep>=0`.


In [ ]:
COLLAPSE_KEYS = ['model', 'mode', 'rep', 'f']
SITE_KEYS = ['model', 'mode', 'rep', 'branch']
required_c = set(COLLAPSE_KEYS + ['P', 'S', 'z'])
required_s = set(SITE_KEYS + ['P', 'S', 'n_sites', 'f1', 'delta_hat', 'predicted_spacing', 'sites'])
for name, frame, required, keys in [('collapse', collapse, required_c, COLLAPSE_KEYS),
                                    ('sites', sites, required_s, SITE_KEYS)]:
    if required - set(frame):
        raise ValueError(f'{name}: colonne mancanti {sorted(required - set(frame))}')
    if frame.empty or frame[keys + ['P', 'S']].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError(f'{name}: tabella vuota, chiavi nulle o chiavi duplicate.')
    if not np.isfinite(frame[['P', 'S']].to_numpy(float)).all() or (frame[['P', 'S']] <= 0).any().any():
        raise ValueError(f'{name}: geometrie non valide.')
    if (frame[['P', 'S', 'rep']] % 1 != 0).any().any():
        raise ValueError(f'{name}: P, S e rep devono essere interi.')
    labels = 'p' + frame.P.astype(int).astype(str) + '-s' + frame.S.astype(int).astype(str)
    if not labels.eq(frame.model.astype(str)).all():
        raise ValueError(f'{name}: nomi modello incoerenti con P e S.')
if not np.isfinite(collapse[['f', 'z']].to_numpy(float)).all() or (collapse.z < 0).any():
    raise ValueError('collapse: frequenze/dispersioni non finite o dispersione negativa.')
if (collapse.rep < 0).any() or not collapse.f.between(*BAND).all():
    raise ValueError('collapse: repliche negative o frequenze fuori dalla banda attuale.')
if not sites.branch.isin(['stride', 'patch']).all() or (sites.rep < -1).any():
    raise ValueError('sites: ramo o indice replica non valido.')
if not np.isfinite(sites.n_sites.to_numpy(float)).all() or (sites.n_sites < 0).any() or (sites.n_sites % 1 != 0).any():
    raise ValueError('sites: conteggi non validi.')
expected_spacing = FS / np.where(sites.branch.eq('stride'), sites.S, sites.P)
if not np.allclose(sites.predicted_spacing, expected_spacing, atol=1e-6):
    raise ValueError('predicted_spacing incompatibile con fs=512 e le geometrie dichiarate.')

def parse_sites(value):
    return np.asarray([float(x) for x in str(value).split()], float) if pd.notna(value) and str(value).strip() else np.array([])

for row in sites.itertuples():
    values = parse_sites(row.sites)
    if len(values) != row.n_sites or not np.isfinite(values).all() or (np.diff(values) < 0).any():
        raise ValueError('sites: lista di frequenze incoerente con n_sites o non ordinata.')
    if row.n_sites >= 1 and not np.isfinite(row.f1):
        raise ValueError('sites: manca f1 per una riga non vuota.')
    if row.n_sites >= 2 and not np.isfinite(row.delta_hat):
        raise ValueError('sites: manca delta_hat per una riga con almeno due siti.')
    if row.n_sites == 0 and (pd.notna(row.f1) or pd.notna(row.delta_hat)):
        raise ValueError('sites: statistiche presenti per una riga vuota.')
    if row.n_sites == 1 and pd.notna(row.delta_hat):
        raise ValueError('sites: delta_hat presente per una riga con un solo sito.')
    if len(values) and not np.isclose(row.f1, values[0], atol=.00051, rtol=0):
        raise ValueError('sites: f1 diverso dal primo sito memorizzato.')

coverage = collapse.groupby(['model', 'P', 'S', 'mode']).agg(
    rows=('f', 'size'), replicas=('rep', 'nunique'), frequencies=('f', 'nunique'),
    f_min=('f', 'min'), f_max=('f', 'max')).reset_index()
expected_pairs = {(f'p{P}-s{S}', mode) for P,S in EXPECTED_MODELS for mode in (*GENERATORS, 'pure')}
observed_pairs = set(zip(coverage.model, coverage['mode']))
if expected_pairs != observed_pairs:
    warning(f'Copertura rispetto al FULL: {len(expected_pairs-observed_pairs)} combinazioni mancanti, '
            f'{len(observed_pairs-expected_pairs)} inattese.')
if not coverage.replicas.eq(3).all():
    warning('Non tutte le curve hanno le tre repliche del FULL.')
for key, group in collapse.groupby(['model', 'mode']):
    grids = [tuple(g.sort_values('f').f) for _, g in group.groupby('rep')]
    if any(grid != grids[0] for grid in grids[1:]):
        raise ValueError(f'{key}: griglie diverse tra repliche; la curva media non è confrontabile.')
TABLES['coverage'] = coverage
display(coverage)


## 3. Funzioni originali e modello D2 conservato

Le funzioni di rilevazione, fusione, assegnazione, sintesi e conteggio sono incorporate dal
repository e non dipendono da import locali. Il controllo successivo confronta la loro ricostruzione
con `02_sites.parquet`: una discrepanza viene mostrata, senza sovrascrivere il file storico.

La factory D2 seguente è identica a quella dei due allegati, inclusa la documentazione originale:

\(y_g\sim N(\kappa_F\Delta_g,\sqrt{\sigma_{extra}^2+\Delta f^2})\),
\(\kappa_F\sim N(0,1)\), \(\sigma_{extra}\sim HalfNormal(5)\).
Nessuna intercetta; risposta primaria `f1`, confronto `delta_hat`, ROPE \(|\kappa_F-1|<0.1\).
Le affermazioni della docstring sul significato di fondamentale e distanza sono proprio oggetto
di diagnosi: qui vengono conservate, non assunte come già verificate.


In [ ]:
def patch_nulls(P: int, fmax: float = BAND[1], fmin: float = BAND[0]) -> list[float]:
    """Patch-integration nulls k*fs/P: a tone completing an integer number of cycles inside one patch."""
    out = [k * FS / P for k in range(1, int(fmax * P / FS) + 2)]
    return [f for f in out if fmin <= f <= fmax]

def stride_locks(S: int, fmax: float = BAND[1], fmin: float = BAND[0]) -> list[float]:
    """Structural stride locks c*fs/S: x[n+S] = x[n], so consecutive patches see identical samples."""
    out = [c * FS / S for c in range(1, int(fmax * S / FS) + 2)]
    return [f for f in out if fmin <= f <= fmax]

def assign_site_family(f: float, P: int, S: int, tol_hz: float = 1.5,
                       ambiguity_hz: float = 0.25) -> tuple[str, float]:
    """Assign an estimated minimum to the nearest predicted grid at sweep resolution.

    Detected minima are merged means and need not equal an analytic grid member to machine
    precision.  A site compatible with both branches is marked ``both`` and set aside by D2, as
    required by Deliverable 3.
    """
    stride = np.asarray(stride_locks(S), dtype=float)
    patch = np.asarray(patch_nulls(P), dtype=float)
    ds = float(np.min(np.abs(stride - f))) if len(stride) else np.inf
    dp = float(np.min(np.abs(patch - f))) if len(patch) else np.inf
    near_s, near_p = ds <= tol_hz, dp <= tol_hz
    if near_s and near_p and (abs(ds - dp) <= ambiguity_hz or max(ds, dp) <= ambiguity_hz):
        return "both", min(ds, dp)
    if near_s and (not near_p or ds + ambiguity_hz < dp):
        return "stride", ds
    if near_p and (not near_s or dp + ambiguity_hz < ds):
        return "patch", dp
    if near_s and near_p:
        return ("stride", ds) if ds < dp else ("patch", dp)
    return "none", min(ds, dp)

def detect_collapse_sites(freqs: np.ndarray, z: np.ndarray, pure: bool,
                          rel_thr: float = 0.02, dip_ratio: float = 0.75,
                          window_hz: float = 6.0) -> list[float]:
    """Frequencies at which the across-patch token dispersion collapses.

    Two regimes, exactly as in hypotheses.py:
      * pure sinusoid , the degeneracy is EXACT, so a site is any frequency whose dispersion is
        within `rel_thr` of zero relative to the curve's own maximum;
      * generator background, the background breaks patch identity, so the dispersion never
        reaches zero; a site is a prominent local minimum dipping to <= `dip_ratio` of its local
        baseline. The baseline window is expressed in Hz (not in samples) because the H3 sweep grid
        is non-uniform.
    """
    freqs = np.asarray(freqs, float)
    z = np.asarray(z, float)
    if pure:
        thr = rel_thr * float(np.nanmax(z))
        return [float(f) for f, v in zip(freqs, z) if v <= thr]

    sites = []
    for i in range(1, len(z) - 1):
        near = (np.abs(freqs - freqs[i]) <= window_hz) & (np.arange(len(freqs)) != i)
        if not near.any():
            continue
        local = float(np.median(z[near]))
        if z[i] <= z[i - 1] and z[i] <= z[i + 1] and local > 0 and z[i] <= dip_ratio * local:
            sites.append(float(freqs[i]))
    return sites

def merge_adjacent(sites: list[float], tol: float = 1.5) -> list[float]:
    """Collapse runs of neighbouring detections into one site (their mean).

    On the union grid a single dip can straddle two nearby grid points (e.g. 42.5 and 42.67 Hz);
    counting both would inflate the site count and corrupt the spacing estimate feeding Eq. (13).
    """
    if not sites:
        return []
    sites = sorted(sites)
    groups, cur = [], [sites[0]]
    for f in sites[1:]:
        if f - cur[-1] <= tol:
            cur.append(f)
        else:
            groups.append(cur)
            cur = [f]
    groups.append(cur)
    return [float(np.mean(g)) for g in groups]

def site_summary(sites: list[float]) -> dict:
    """The two derived statistics the H3 movement model (Eq. 14) regresses on.

    `f1` is the lowest detected site (the comb's fundamental) and `delta_hat` the median successive
    difference (its spacing). Both are read off the data WITHOUT assuming which parameter generates
    the comb, that is the point: the regression then estimates whether the spacing moves as 1/S.
    """
    sites = sorted(sites)
    if not sites:
        return {"n_sites": 0, "f1": np.nan, "delta_hat": np.nan}
    if len(sites) == 1:
        # a single in-band site is its own fundamental; the spacing is unidentified from differences
        return {"n_sites": 1, "f1": float(sites[0]), "delta_hat": np.nan}
    diffs = np.diff(sites)
    return {"n_sites": len(sites), "f1": float(sites[0]), "delta_hat": float(np.median(diffs))}

pl = SimpleNamespace(FS=FS, detect_collapse_sites=detect_collapse_sites,
    merge_adjacent=merge_adjacent, assign_site_family=assign_site_family, site_summary=site_summary)

def derive_sites(collapse: pd.DataFrame, cfg: Config | None = None) -> pd.DataFrame:
    """Detected collapse sites, split by branch, for the H3 movement models.

    Deliverable 3, H3: "Every detected dip is assigned to the branch that predicts it, and sites
    belonging to both are set aside. The fundamental of branch F is then compared with the spacing
    that branch predicts." A row is therefore (geometry, signal mode, replicate, BRANCH) and it
    carries the fundamental of that branch only. A single pooled spacing would be meaningless:
    F_lock is a union, and the union of two combs has two interlaced spacings rather than one.

    Sites are detected per replicate (giving the model its residual variance) and once more on the
    replicate-averaged curve, recorded as rep = -1.
    """
    cfg = cfg or Config()
    rows = []
    for (model, mode), g in collapse.groupby(["model", "mode"]):
        P, S = int(g["P"].iloc[0]), int(g["S"].iloc[0])
        pieces = [(r, sub) for r, sub in g.groupby("rep")]
        mean_curve = g.groupby("f", as_index=False)["z"].mean()
        pieces.append((-1, mean_curve))
        for rep, sub in pieces:
            sub = sub.sort_values("f")
            sites = pl.detect_collapse_sites(sub["f"].to_numpy(), sub["z"].to_numpy(),
                                             pure=(mode == "pure"))
            sites = pl.merge_adjacent(sites, tol=cfg.site_merge_tol_hz)

            # assign every detected site to the branch that predicts it; "both" is set aside
            by_branch: dict[str, list[float]] = {"stride": [], "patch": []}
            n_both = 0
            n_unassigned = 0
            assignment_residuals: dict[str, list[float]] = {"stride": [], "patch": []}
            for s in sites:
                fam, residual = pl.assign_site_family(
                    s,
                    P,
                    S,
                    tol_hz=cfg.site_assignment_tol_hz,
                    ambiguity_hz=cfg.site_ambiguity_hz,
                )
                if fam == "both":
                    n_both += 1
                elif fam in by_branch:
                    by_branch[fam].append(s)
                    assignment_residuals[fam].append(residual)
                else:
                    n_unassigned += 1

            for branch, members in by_branch.items():
                rows.append(dict(model=model, P=P, S=S, mode=mode, rep=int(rep),
                                 branch=branch,
                                 predicted_spacing=pl.FS / (S if branch == "stride" else P),
                                 sites=" ".join(f"{s:.3f}" for s in members),
                                 n_ambiguous=n_both, n_unassigned=n_unassigned,
                                 max_assignment_error=(max(assignment_residuals[branch])
                                                       if assignment_residuals[branch] else np.nan),
                                 **pl.site_summary(members)))
    return pd.DataFrame(rows)

def identified_branches(
    sites: pd.DataFrame, minimum_sites: int = 10, modes: tuple[str, ...] = ("tsmixup", "kernelsynth")
) -> dict[str, bool]:
    """Apply Deliverable 3's minimum-ten-unambiguous-sites bar to mean collapse curves."""
    required = {"branch", "n_sites", "rep", "mode"}
    if sites.empty or not required.issubset(sites.columns):
        return {"stride": False, "patch": False}
    mean_curves = sites[(sites["rep"] == -1) & sites["mode"].isin(modes)]
    totals = mean_curves.groupby("branch")["n_sites"].sum()
    return {branch: int(totals.get(branch, 0)) >= minimum_sites for branch in ("stride", "patch")}


In [ ]:
def model_D2_movement(df: pd.DataFrame, branch: str, resolution_hz: float | None = None,
                      response: str = "f1") -> pm.Model:
    """Models D2, H3a and H3b.  One scaling law per branch, no intercept.

        f1_hat[g] ~ Normal(kappa_F * Delta_F[g], sqrt(sigma_F^2 + df^2))

    Deliverable 3, H3: every detected dip is first assigned to the branch that predicts it and
    ambiguous sites are set aside, then the fundamental of that branch is compared with the spacing
    that branch predicts, Delta_S = fs/S or Delta_P = fs/P. H3a predicts kappa_S = 1 and H3b
    predicts kappa_P = 1: each branch tracks its own parameter one for one.

    This replaced a single regression of one pooled spacing on BOTH predictors, with an intercept
    and the prediction kappa_P = 0. That formulation contradicted H3 as approved in the approved Deliverable 3 formulation,
    which claims that each branch moves with its own parameter; and it treated the detected set as
    one comb, when F_lock is a union and the union of two combs has two interlaced spacings. The
    dip that formulation had to call spurious at p16-s8 is a patch null at fs/P = 32 Hz, i.e. an
    observation H3 predicts.

    **Measurement resolution.** The spacing is read off a frequency sweep, so it carries a floor of
    about one grid step whatever the sampling noise. Stating it is not cosmetic: when the sites land
    on the predicted comb exactly, the residuals vanish, sigma is pushed to zero and the posterior
    develops a funnel NUTS cannot traverse (R-hat above 2, hundreds of divergences). Adding the
    floor in quadrature removes the pathology and is the honest statement: a near-zero sigma is then
    a result, the spacings follow the law to within the sweep resolution, not a sampler failure.

    `response="f1"` uses the branch's fundamental, the lowest detected site of that branch, which is
    what the deliverable specifies. `response="delta_hat"` uses the median gap of the same branch and
    is fitted only as a robustness check: with three or four sites in band one spurious detection
    inserts a short interval and flips the median, to which the fundamental is immune.
    """
    sub = df[df["branch"] == branch]
    y = sub[response].to_numpy(float)                      # measured spacing [Hz]
    x = sub["predicted_spacing"].to_numpy(float)           # fs/S or fs/P [Hz]
    name = {"stride": "kappa_S", "patch": "kappa_P"}[branch]

    step = resolution_hz if resolution_hz is not None else CFG.collapse_step
    with pm.Model(coords={"obs": np.arange(len(y))}) as m:
        kappa = pm.Normal(name, mu=0.0, sigma=1.0)         # H3a / H3b predict 1
        sigma_d = pm.HalfNormal("sigma_extra", 5.0)        # scatter beyond the grid floor [Hz]
        sigma_total = pm.math.sqrt(sigma_d ** 2 + step ** 2)
        pm.Normal("f1_hat", mu=kappa * x, sigma=sigma_total, observed=y, dims="obs")
    return m

D2_FACTORY_AST_SHA256 = '24db4c1d2a9a6a3de92aa40aced19b35f078a7c1664ca0f7795a0d7be99c0b3c'
PROVENANCE["D2_factory_AST_sha256"] = D2_FACTORY_AST_SHA256
print("Factory D2 conservata; nessun fit avviato.")


## 4. Riproduzione del gate e confronto con i siti salvati

Il gate resta quello attuale: almeno dieci siti non ambigui sulle curve medie generate.
Non è un conteggio di frequenze globalmente distinte: la stessa frequenza in geometrie o generatori
diversi contribuisce più volte. Un gate numerico superato non sostituisce la verifica del disegno.


In [ ]:
mean_saved = sites[(sites.rep == -1) & sites['mode'].isin(GENERATORS)].copy()
by_generator = mean_saved.groupby(['mode', 'branch']).n_sites.sum().unstack('branch', fill_value=0)
totals = mean_saved.groupby('branch').n_sites.sum().reindex(['stride', 'patch'], fill_value=0)
D2_IDENTIFICATION = identified_branches(sites, CFG.min_d2_sites, GENERATORS)
gate = pd.DataFrame({'branch': totals.index, 'sites': totals.values,
                     'minimum': CFG.min_d2_sites,
                     'identified': [D2_IDENTIFICATION[b] for b in totals.index]})
TABLES['gate'] = gate
display(by_generator)
display(gate)
patch_saved = mean_saved[(mean_saved.branch == 'patch') & (mean_saved.n_sites > 0)]
display(patch_saved[['model', 'mode', 'n_sites', 'sites', 'f1', 'delta_hat']])

sites_rederived = derive_sites(collapse, CFG)
comparison = sites.merge(sites_rederived, on=SITE_KEYS, how='outer', suffixes=('_saved', '_replay'),
                         indicator=True, validate='one_to_one')
numeric = ['P', 'S', 'n_sites', 'f1', 'delta_hat', 'predicted_spacing',
           'n_ambiguous', 'n_unassigned', 'max_assignment_error']
checks = []
for col in numeric:
    if col + '_saved' in comparison and col + '_replay' in comparison:
        flag = np.isclose(comparison[col+'_saved'].to_numpy(float),
                          comparison[col+'_replay'].to_numpy(float), atol=1e-6, rtol=1e-7, equal_nan=True)
        comparison[col+'_matches'] = flag
        checks.append(col+'_matches')
def same_sites(row):
    a, b = parse_sites(row.sites_saved), parse_sites(row.sites_replay)
    return len(a) == len(b) and np.allclose(a, b, atol=.00051, rtol=0)
comparison['sites_matches'] = comparison.apply(same_sites, axis=1)
comparison['matches'] = comparison['_merge'].eq('both') & comparison[checks + ['sites_matches']].all(axis=1)
REPLAY_MATCHES = bool(comparison.matches.all())
TABLES['sites_rederived'] = sites_rederived
TABLES['sites_reconciliation'] = comparison.astype({'_merge': str})
print(f'Riproduzione dei siti: {comparison.matches.sum()}/{len(comparison)} righe coincidenti.')
if not REPLAY_MATCHES:
    warning('I siti salvati non sono riprodotti dal rilevatore incorporato: controllare versione/configurazione prima di interpretare la selezione.')
    display(comparison.loc[~comparison.matches, SITE_KEYS + ['n_sites_saved', 'n_sites_replay',
                                                           'f1_saved', 'f1_replay']].head(30))
fig, ax = plt.subplots(figsize=(6, 3.5), layout='constrained')
ax.bar(gate.branch, gate.sites, color=['#6a51a3', '#2171b5'])
ax.axhline(CFG.min_d2_sites, color='#b2182b', ls='--', label='Soglia attuale: 10')
for i, n in enumerate(gate.sites): ax.annotate(str(n), (i, n), xytext=(0, 5), textcoords='offset points', ha='center')
ax.set(ylim=(0, max(12, gate.sites.max()*1.2)), ylabel='Siti contati sulle curve medie',
       title='Gate D2 dai Parquet salvati, background generati')
ax.legend()
FIGURES.append(('D2_site_gate', fig))
plt.show()


## 5. Traccia completa: punti, minimi, fusione, assegnazione

Per ogni curva si registrano tutti i punti, il rapporto rispetto alla mediana locale, il test
di minimo locale, i siti prima/dopo fusione e il ramo assegnato. I punti iniziale e finale
sono esclusi dal rilevatore per background generati, come nel codice originale.
Per `pure` vale invece la soglia del 2% del massimo globale: i due criteri non sono intercambiabili.


In [ ]:
CURVES = {}
point_rows, merged_rows, funnel_rows = [], [], []
for (model, mode), group in collapse.groupby(['model', 'mode']):
    P, S = int(group.P.iloc[0]), int(group.S.iloc[0])
    pieces = [(int(rep), g[['f','z']]) for rep,g in group.groupby('rep')]
    pieces.append((-1, group.groupby('f', as_index=False).z.mean()))
    for rep, sub in pieces:
        sub = sub.sort_values('f').reset_index(drop=True)
        f, z = sub.f.to_numpy(float), sub.z.to_numpy(float)
        raw = detect_collapse_sites(f, z, pure=mode=='pure')
        merged = merge_adjacent(raw, CFG.site_merge_tol_hz)
        meta = dict(model=model, P=P, S=S, mode=mode, rep=rep)
        audited = []
        for i in range(len(f)):
            near = (abs(f-f[i]) <= DETECTOR['window_hz']) & (np.arange(len(f)) != i)
            baseline = float(np.median(z[near])) if near.any() else np.nan
            interior = 0 < i < len(f)-1
            is_min = bool(interior and z[i] <= z[i-1] and z[i] <= z[i+1])
            ratio = z[i]/baseline if baseline > 0 else np.nan
            passed = bool(z[i] <= DETECTOR['rel_thr']*z.max()) if mode=='pure' else bool(
                interior and near.any() and is_min and baseline > 0 and z[i] <= DETECTOR['dip_ratio']*baseline)
            point_rows.append(dict(**meta, f=f[i], z=z[i], baseline=baseline, ratio=ratio,
                                   interior=interior, local_minimum=is_min, detected=passed,
                                   pure_threshold=DETECTOR['rel_thr']*z.max()))
            if passed: audited.append(float(f[i]))
        if audited != raw:
            raise AssertionError('La traccia diagnostica non coincide con il rilevatore originale.')
        counts = dict(stride=0, patch=0, both=0, none=0)
        for value in merged:
            family, error = assign_site_family(value, P, S, CFG.site_assignment_tol_hz, CFG.site_ambiguity_hz)
            counts[family] += 1
            merged_rows.append(dict(**meta, f=value, family=family, assignment_error=error))
        funnel_rows.append(dict(**meta, sampled_points=len(f), raw_detections=len(raw),
                                merged_detections=len(merged), **counts))
        CURVES[(model, mode, rep)] = dict(P=P, S=S, f=f, z=z, raw=raw, merged=merged)

points = pd.DataFrame(point_rows)
merged_sites = pd.DataFrame(merged_rows, columns=['model','P','S','mode','rep','f','family','assignment_error'])
funnel = pd.DataFrame(funnel_rows)
TABLES.update(point_audit=points, merged_sites_audit=merged_sites, detection_stages=funnel)
display(funnel[(funnel.rep == -1) & funnel['mode'].isin(GENERATORS)])
print('both/none sono contati una sola volta per curva, senza duplicare le righe dei due rami.')


## 6. Frequenze teoriche patch: perché passano o non passano?

Si esaminano le frequenze patch teoriche **senza aggiungerle ai siti rilevati**. Per ciascuna si
mostrano il campione più vicino, l'eventuale minimo locale entro la tolleranza e il risultato
della selezione originale. Il motivo è un resoconto del filtro, non una prova dell'assenza fisica
del fenomeno. Anche i siti teorici condivisi sono inclusi e marcati `both`.


In [ ]:
candidate_rows = []
for (model, mode, rep), curve in CURVES.items():
    P, S, f = curve['P'], curve['S'], curve['f']
    audit = points[(points.model==model) & (points['mode']==mode) & (points.rep==rep)]
    detected = merged_sites[(merged_sites.model==model) & (merged_sites['mode']==mode) & (merged_sites.rep==rep)]
    for target in patch_nulls(P):
        nearest = audit.iloc[int(np.argmin(abs(f-target)))]
        theoretical_family = assign_site_family(target, P, S, CFG.site_assignment_tol_hz, CFG.site_ambiguity_hz)[0]
        close_points = audit[abs(audit.f-target) <= CFG.site_assignment_tol_hz]
        close_minima = close_points[close_points.local_minimum]
        close_detected = detected[abs(detected.f-target) <= CFG.site_assignment_tol_hz]
        best_f = np.nan
        best_ratio = np.nan
        if len(close_minima):
            best = close_minima.sort_values(['ratio','f'], na_position='last').iloc[0]
            best_f, best_ratio = best.f, best.ratio
        detected_f = np.nan
        detected_family = ''
        if len(close_detected):
            chosen = close_detected.iloc[np.argmin(abs(close_detected.f.to_numpy()-target))]
            detected_f, detected_family = chosen.f, chosen.family
            reason = 'retained_' + str(chosen.family)
        elif close_points.empty:
            reason = 'no_sample_within_tolerance'
        elif close_points.detected.any():
            reason = 'lost_after_merge'
        elif mode == 'pure':
            reason = 'above_pure_threshold'
        elif not close_points.interior.any():
            reason = 'endpoint_excluded'
        elif close_minima.empty:
            reason = 'no_local_minimum'
        else:
            reason = 'insufficient_depth_or_baseline'
        candidate_rows.append(dict(model=model, P=P, S=S, mode=mode, rep=rep,
                                    target_hz=target, theoretical_family=theoretical_family,
                                    sample_hz=nearest.f, sample_offset_hz=abs(nearest.f-target),
                                    sample_z=nearest.z, sample_ratio=nearest.ratio,
                                    local_min_hz=best_f, local_min_ratio=best_ratio,
                                    detected_hz=detected_f, detected_family=detected_family, reason=reason))
patch_candidates = pd.DataFrame(candidate_rows)
TABLES['patch_candidates'] = patch_candidates
exclusive = patch_candidates[(patch_candidates.rep == -1) & patch_candidates.theoretical_family.eq('patch')]
candidate_summary = exclusive.groupby(['mode','reason']).size().rename('candidate_count').reset_index()
TABLES['patch_candidate_reasons'] = candidate_summary
display(candidate_summary)
display(exclusive[exclusive['mode'].isin(GENERATORS)].sort_values('local_min_ratio').head(20))
print('I conteggi dei candidati teorici non sono osservazioni D2 e non vengono usati per superare il gate.')


## 7. Stabilità tra repliche e sito storico a 240 Hz

Un sito della curva media è confrontato con i siti rilevati nelle singole repliche, con la stessa
tolleranza di assegnazione e lo stesso ramo. Si riporta anche quante rilevazioni delle repliche
non hanno un sito dello stesso ramo nella curva media. La quota di repliche è descrittiva,
non una probabilità posteriore; le repliche non vengono sommate al conteggio del gate.


In [ ]:
support_rows = []
for row in merged_sites[merged_sites.rep == -1].itertuples():
    reps = sorted(collapse[(collapse.model==row.model) & (collapse['mode']==row.mode)].rep.unique())
    hits, locations = [], []
    for rep in reps:
        g = merged_sites[(merged_sites.model==row.model) & (merged_sites['mode']==row.mode)
                         & (merged_sites.rep==rep) & merged_sites.family.eq(row.family)]
        close = g[abs(g.f-row.f) <= CFG.site_assignment_tol_hz]
        if len(close):
            hits.append(int(rep))
            locations.append(float(close.iloc[np.argmin(abs(close.f.to_numpy()-row.f))].f))
    support_rows.append(dict(model=row.model, mode=row.mode, family=row.family, mean_site_hz=row.f,
                             n_replicas=len(reps), supported_replicas=len(hits),
                             support_fraction=len(hits)/len(reps), replicas=' '.join(map(str,hits)),
                             replica_frequencies=' '.join(f'{f:.6f}' for f in locations)))
support = pd.DataFrame(support_rows, columns=['model','mode','family','mean_site_hz','n_replicas',
                                             'supported_replicas','support_fraction','replicas','replica_frequencies'])
TABLES['replicate_support'] = support
display(support[support.family.eq('patch')])

replica_only_rows = []
for row in merged_sites[(merged_sites.rep>=0) & merged_sites.family.eq('patch')].itertuples():
    mean = merged_sites[(merged_sites.model==row.model) & (merged_sites['mode']==row.mode)
                        & (merged_sites.rep==-1) & merged_sites.family.eq('patch')]
    if not (abs(mean.f-row.f) <= CFG.site_assignment_tol_hz).any():
        replica_only_rows.append(dict(model=row.model, mode=row.mode, rep=row.rep, site_hz=row.f))
replica_only = pd.DataFrame(replica_only_rows, columns=['model','mode','rep','site_hz'])
TABLES['patch_replica_only'] = replica_only
print('Rilevazioni patch nelle repliche senza corrispondenza patch nella media:', len(replica_only))
display(replica_only.head(20))

historical_target = patch_candidates[(patch_candidates.model=='p32-s16')
                                    & (patch_candidates['mode']=='tsmixup')
                                    & np.isclose(patch_candidates.target_hz, 240)]
print('Controllo del sito storico: p32-s16 / tsmixup / 240 Hz')
display(historical_target[['rep','target_hz','sample_z','local_min_ratio','detected_hz','reason']])


## 8. Curve e soglie a confronto

I pannelli mostrano la curva media e le singole repliche, le due griglie e le rilevazioni
dopo fusione. I grafici coprono casi prefissati e le geometrie con siti patch medi rilevati;
la tabella completa rimane disponibile anche per i casi non mostrati.


In [ ]:
COLORS = {'stride':'#6a51a3', 'patch':'#2171b5', 'both':'#555555', 'none':'#d95f02'}
def plot_curve(model, mode, zoom=None):
    key = (model, mode, -1)
    if key not in CURVES:
        return
    c = CURVES[key]
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, layout='constrained')
    for (m, md_, rep), r in CURVES.items():
        if m==model and md_==mode and rep>=0:
            axes[0].plot(r['f'], r['z'], color='0.65', alpha=.5, lw=.8)
    axes[0].plot(c['f'], c['z'], color='#b2182b', lw=1.5, label='Curva media')
    audit = points[(points.model==model) & (points['mode']==mode) & (points.rep==-1)]
    if mode=='pure':
        axes[0].axhline(float(audit.pure_threshold.iloc[0]), color='#008837', ls='--', label='Soglia pure: 2% del massimo')
        axes[1].plot(audit.f, audit.z / max(float(audit.z.max()), 1e-15), color='#b2182b')
        axes[1].axhline(.02, color='#008837', ls='--')
        axes[1].set_ylabel('z / massimo globale')
    else:
        axes[0].plot(audit.f, .75*audit.baseline, color='#008837', ls='--', lw=1, label='0,75 × mediana locale')
        axes[1].plot(audit.f, audit.ratio, color='#b2182b')
        axes[1].axhline(.75, color='#008837', ls='--')
        axes[1].set_ylabel('z / mediana locale')
    for branch, grid, style in [('stride',stride_locks(c['S']),'--'), ('patch',patch_nulls(c['P']),':')]:
        for j, f in enumerate(grid):
            axes[0].axvline(f, color=COLORS[branch], ls=style, lw=.8, alpha=.6,
                           label=f'Griglia {branch}' if j==0 else None)
    g = merged_sites[(merged_sites.model==model) & (merged_sites['mode']==mode) & (merged_sites.rep==-1)]
    for family in COLORS:
        freq = g[g.family==family].f.to_numpy(float)
        if len(freq):
            axes[0].scatter(freq, np.interp(freq,c['f'],c['z']), s=35, marker='x',
                            color=COLORS[family], label=f'Rilevati: {family}', zorder=5)
    axes[0].set(ylabel='Dispersione token z', title=f'{model} / {mode}: curva media, repliche e rilevazioni')
    axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=9)
    axes[1].set_xlabel('Frequenza [Hz]')
    if zoom: axes[1].set_xlim(*zoom)
    FIGURES.append((f'curve_{model}_{mode}' + ('_zoom' if zoom else ''), fig))
    plt.show()

plot_targets = [('p16-s8','tsmixup'), ('p16-s8','pure'), ('p32-s16','tsmixup'), ('p32-s16','pure')]
plot_targets += list(zip(patch_saved.model, patch_saved['mode']))
for model, mode in list(dict.fromkeys(plot_targets))[:12]:
    plot_curve(model, mode)
plot_curve('p32-s16', 'tsmixup', zoom=(228,250))
if not FIGURES and CURVES:
    model, mode, _ = next(iter(CURVES))
    plot_curve(model, mode)


## 9. Fondamentale, armoniche mancanti e distanze

Il modello attuale usa il primo sito rilevato come `f1`. Qui si controlla se tale frequenza sia
compatibile con l'armonica 1 o con un'armonica successiva. Si confrontano anche le distanze
successive con le differenze tra gli indici armonici. **Nessuna risposta viene corretta o sostituita.**

L'eliminazione dei siti condivisi può rendere `delta_hat` diversa da \(f_s/S\) anche con un pettine
stride ideale. Per esempio, per P=24 e S=16, eliminando 64, 128, 192 Hz restano
32, 96, 160, 224 Hz: la distanza è 64 Hz, mentre \(f_s/S=32\) Hz.


In [ ]:
harmonic_rows, gap_rows, geometry_rows = [], [], []
for row in sites[sites.n_sites > 0].itertuples():
    freq = parse_sites(row.sites)
    delta = float(row.predicted_spacing)
    harmonic = np.rint(freq/delta).astype(int)
    residual = freq - harmonic*delta
    harmonic_rows.append(dict(model=row.model, mode=row.mode, rep=row.rep, branch=row.branch,
        n_sites=row.n_sites, predicted_spacing=delta, f1=row.f1, f1_over_spacing=row.f1/delta,
        first_harmonic=int(harmonic[0]), fundamental_detected=bool(harmonic[0]==1 and abs(residual[0])<=CFG.site_assignment_tol_hz),
        max_harmonic_error=float(abs(residual).max()), delta_hat=row.delta_hat,
        gap_over_spacing=row.delta_hat/delta if pd.notna(row.delta_hat) else np.nan))
    for j in range(len(freq)-1):
        gap_rows.append(dict(model=row.model, mode=row.mode, rep=row.rep, branch=row.branch,
            left_hz=freq[j], right_hz=freq[j+1], gap_hz=freq[j+1]-freq[j],
            harmonic_jump=int(harmonic[j+1]-harmonic[j]),
            predicted_gap_hz=(harmonic[j+1]-harmonic[j])*delta))
for P,S in sorted(set(zip(collapse.P.astype(int),collapse.S.astype(int)))):
    for branch, grid, delta in [('stride',stride_locks(S),FS/S), ('patch',patch_nulls(P),FS/P)]:
        kept = [f for f in grid if assign_site_family(f,P,S,CFG.site_assignment_tol_hz,CFG.site_ambiguity_hz)[0]==branch]
        geometry_rows.append(dict(model=f'p{P}-s{S}', branch=branch, predicted_spacing=delta,
            total_theoretical_sites=len(grid), exclusive_theoretical_sites=len(kept),
            first_exclusive_hz=kept[0] if kept else np.nan,
            exclusive_median_gap=float(np.median(np.diff(kept))) if len(kept)>1 else np.nan))
harmonics = pd.DataFrame(harmonic_rows, columns=['model','mode','rep','branch','n_sites','predicted_spacing',
    'f1','f1_over_spacing','first_harmonic','fundamental_detected','max_harmonic_error','delta_hat','gap_over_spacing'])
gaps = pd.DataFrame(gap_rows, columns=['model','mode','rep','branch','left_hz','right_hz','gap_hz','harmonic_jump','predicted_gap_hz'])
geometry = pd.DataFrame(geometry_rows)
TABLES.update(harmonic_audit=harmonics, gap_audit=gaps, theoretical_geometry=geometry)
display(harmonics[(harmonics.rep==-1) & harmonics['mode'].isin(GENERATORS)])
display(geometry[geometry.model.isin(['p24-s16','p32-s16'])])

fig, axes = plt.subplots(1,2,figsize=(11,4),layout='constrained')
for branch in ('stride','patch'):
    g = harmonics[(harmonics.rep>=0) & harmonics['mode'].isin(GENERATORS) & harmonics.branch.eq(branch)]
    for ax, response in zip(axes, ['f1','delta_hat']):
        valid = g[g[response].notna()]
        ax.scatter(valid.predicted_spacing, valid[response], s=32, alpha=.5, color=COLORS[branch], label=branch)
for ax, response in zip(axes,['f1','delta_hat']):
    ax.plot([0,FS/8],[0,FS/8], color='0.4',ls='--',label='y = spacing teorico')
    ax.set(xlabel='Spacing teorico [Hz]', ylabel=f'{response} salvato [Hz]', title=f'Input D2: {response}, repliche generate')
    ax.legend()
FIGURES.append(('D2_inputs',fig))
plt.show()


## 10. Controllo geometrico della misura

Questa dimostrazione deterministica non usa Chronos e non produce nuove osservazioni.
Con P=32, S=16 si confrontano due patch consecutive di una sinusoide pura:
la media di entrambe può essere nulla anche quando i vettori sono diversi.
Il Parquet contiene la dispersione degli embedding, non le medie dei campioni; questo controllo
spiega la distinzione, ma non dimostra da solo la causa dei minimi mancanti nell'embedding appreso.


In [ ]:
demo_rows = []
P, S = 32, 16
for freq in (16.,32.,240.):
    x = np.sin(2*np.pi*freq*np.arange(P+S)/FS + .3)
    a, b = x[:P], x[S:S+P]
    demo_rows.append(dict(f_hz=freq, patch_mean_1=a.mean(), patch_mean_2=b.mean(),
                          max_sample_difference=np.max(abs(a-b)),
                          same_samples=bool(np.allclose(a,b,atol=1e-12))))
geometry_demo = pd.DataFrame(demo_rows)
TABLES['deterministic_patch_demo'] = geometry_demo
display(geometry_demo)


## 11. Input del modello originale e costruzione opzionale

I filtri sono gli stessi degli allegati: `rep>=0`, background generati e almeno un sito per `f1`;
almeno due e `delta_hat` non nullo per il confronto. I rami sotto soglia non vengono costruiti.
`BUILD_MODEL_GRAPH=True` controlla solo che la factory possa costruire il modello sugli input.
Questo notebook non contiene chiamate a `pm.sample` e non legge o riscrive posteriori.


In [ ]:
sites_f1 = sites[(sites.rep >= 0) & (sites.n_sites >= 1) & sites['mode'].isin(GENERATORS)].copy()
sites_gap = sites[(sites.rep >= 0) & (sites.n_sites >= 2) & sites.delta_hat.notna()
                  & sites['mode'].isin(GENERATORS)].copy()
input_rows = []
for branch in ('stride','patch'):
    for response, frame in [('f1',sites_f1),('delta_hat',sites_gap)]:
        sub = frame[frame.branch==branch]
        input_rows.append(dict(branch=branch, response=response, rows=len(sub),
                                geometries=sub.model.nunique(), predictor_values=sub.predicted_spacing.nunique(),
                                gate_passed=D2_IDENTIFICATION[branch]))
        if len(sub) and not np.isfinite(sub[[response,'predicted_spacing']].to_numpy(float)).all():
            raise ValueError('Input D2 non finiti.')
        if len(sub) and sub.predicted_spacing.nunique()<2:
            warning(f'{branch}/{response}: un solo valore del predittore; il movimento tra geometrie non è verificabile.')
input_summary = pd.DataFrame(input_rows)
TABLES['model_inputs'] = input_summary
display(input_summary)
MODEL_GRAPH_STATUS = 'non richiesto'
if BUILD_MODEL_GRAPH:
    import pymc as pm
    for branch in ('stride','patch'):
        if not D2_IDENTIFICATION[branch]:
            print(f'{branch}: NOT IDENTIFIED; modello non costruito.')
            continue
        for response, frame in [('f1',sites_f1), ('delta_hat',sites_gap)]:
            if len(frame[frame.branch==branch]):
                model = model_D2_movement(frame, branch, response=response)
                print(branch, response, model)
    MODEL_GRAPH_STATUS = 'costruzione completata, nessun campionamento'


## 12. Sensibilità della rilevazione: soglie alternative, D2 invariato

L'esecuzione precedente sui dati FULL ha confermato 67 siti stride e 1 patch. La nuova domanda è:
**allentare la soglia recupera depressioni patch distinguibili dai controlli, oppure aumenta anche
le rilevazioni altrove?** I risultati di questa sezione sono esplorativi sullo stesso campione,
non una validazione indipendente di una nuova soglia.

Il riferimento resta `dip_ratio=0.75`, cioè profondità locale almeno del 25%. A 0.80 basta il 20%,
a 0.90 il 10%. Si esplora la griglia fissa 0.60–0.95 a passi di 0.05, senza scegliere automaticamente
un valore. La funzione originale viene richiamata con il solo parametro `dip_ratio` variato;
fusione, assegnazione e tolleranze restano uguali. La curva media viene ricalcolata come prima.
Le sinusoidi pure mantengono il loro diverso criterio del 2% del massimo globale e non entrano
in questa sensibilità.

Le tabelle alternative sono separate da `sites`, `sites_f1`, `sites_gap` e `D2_IDENTIFICATION`.
Superare dieci siti in una variante è indicato come **solo conteggio**, non come identificazione
scientifica o autorizzazione a eseguire D2. Ulteriori rilevazioni possono fondersi tra loro:
il conteggio finale non deve essere assunto monotono al crescere della soglia.

In [ ]:
THRESHOLD_GRID = (.60, .65, .70, .75, .80, .85, .90, .95)
REFERENCE_THRESHOLD = .75
CONTROL_OFFSETS_HZ = (4.0, 6.0)
CONTROL_SAMPLE_TOL_HZ = .51
TARGET_SAMPLE_TOL_HZ = .001
CONTROL_WINDOW_HZ = CFG.site_assignment_tol_hz
BASELINE_D2_GATE = dict(D2_IDENTIFICATION)
BASELINE_SITES_HASH = hashlib.sha256(pd.util.hash_pandas_object(sites, index=True).values.tobytes()).hexdigest()
alternative_rows, alternative_site_rows = [], []
for (model, mode, rep), curve in CURVES.items():
    if mode not in GENERATORS:
        continue
    for threshold in THRESHOLD_GRID:
        detected = detect_collapse_sites(curve['f'], curve['z'], pure=False,
                                          dip_ratio=threshold,
                                          window_hz=DETECTOR['window_hz'])
        combined = merge_adjacent(detected, CFG.site_merge_tol_hz)
        family_counts = dict(stride=0, patch=0, both=0, none=0)
        for frequency in combined:
            family, error = assign_site_family(frequency, curve['P'], curve['S'],
                                                CFG.site_assignment_tol_hz, CFG.site_ambiguity_hz)
            family_counts[family] += 1
            alternative_site_rows.append(dict(model=model, mode=mode, rep=rep, threshold=threshold,
                                                frequency_hz=frequency, family=family,
                                                assignment_error=error))
        alternative_rows.append(dict(model=model, mode=mode, rep=rep, threshold=threshold,
                                       raw_detections=len(detected), merged_detections=len(combined),
                                       **family_counts))
        if threshold == REFERENCE_THRESHOLD:
            original = merged_sites[(merged_sites.model==model) & (merged_sites['mode']==mode)
                                     & (merged_sites.rep==rep)].sort_values('f')
            if not np.array_equal(np.asarray(combined), original.f.to_numpy()):
                raise AssertionError('La variante 0.75 non riproduce le rilevazioni originali.')
threshold_curves = pd.DataFrame(alternative_rows)
threshold_sites = pd.DataFrame(alternative_site_rows,
    columns=['model','mode','rep','threshold','frequency_hz','family','assignment_error'])
count_columns = ['raw_detections','merged_detections','stride','patch','both','none']
threshold_totals = threshold_curves[threshold_curves.rep==-1].groupby('threshold')[count_columns].sum().reset_index()
threshold_totals['patch_passes_count_only'] = threshold_totals.patch >= CFG.min_d2_sites
threshold_totals['stride_passes_count_only'] = threshold_totals.stride >= CFG.min_d2_sites
threshold_by_generator = threshold_curves[threshold_curves.rep==-1].groupby(['threshold','mode'])[count_columns].sum().reset_index()
TABLES.update(threshold_curve_counts=threshold_curves, threshold_sites=threshold_sites,
              threshold_totals=threshold_totals, threshold_by_generator=threshold_by_generator)
PROVENANCE['threshold_sensitivity'] = dict(thresholds=THRESHOLD_GRID, reference=REFERENCE_THRESHOLD,
                                           exploratory=True, D2_inputs_unchanged=True)
display(threshold_totals)
display(threshold_by_generator)
print('none indica minimi fuori dalle griglie: non sono automaticamente falsi positivi.')

## 13. Controlli locali scelti solo dalle frequenze

Per ogni candidato **patch esclusivo** si scelgono due controlli simmetrici, prima a ±4 Hz e
separatamente a ±6 Hz. Il campione più vicino al controllo deve distare al massimo 0.51 Hz dal
punto richiesto; il candidato deve avere un campione entro 0.001 Hz. Si registrano gli scarti reali.
La coppia è utilizzabile solo se entrambe le finestre di controllo (±1.5 Hz) sono interamente
nella banda e tutti i loro campioni distano **più di 1.5 Hz da entrambe le griglie teoriche**.
Non si spostano i controlli per farli passare e non si sceglie l'offset in funzione dei risultati.
Le esclusioni restano nella tabella di copertura.

Due confronti distinti:

1. **Profondità continua al punto:** $d=1-z/\mathrm{mediana\ locale}$. Si confronta il candidato
   con la media dei suoi due controlli. Un delta positivo indica una depressione maggiore al
   candidato. Nessuna ricerca del minimo favorisce il candidato in questo confronto.
2. **Rilevabilità nella finestra:** si cerca un minimo locale entro ±1.5 Hz sia al candidato sia
   a ciascun controllo, usando la stessa soglia. Si confrontano le quote di finestre con almeno
   una rilevazione. È un confronto **prima della fusione e assegnazione**, distinto dal gate D2.

Le frequenze di controllo non sono garantite prive di effetti del modello. La loro quota di
rilevazioni è un riferimento locale, non una stima certificata dei falsi positivi.
Geometrie, frequenze e repliche correlate non sono trattate come osservazioni indipendenti:
si mostrano riepiloghi descrittivi per geometria e generatore, senza p-value o nuove posteriori.

In [ ]:
paired_rows = []
def grid_distance(values, P, S):
    theoretical = np.unique(np.r_[patch_nulls(P), stride_locks(S)])
    return np.min(abs(np.asarray(values)[:, None]-theoretical[None, :]), axis=1)

def window_score(audit, center):
    eligible = audit[(abs(audit.f-center) <= CONTROL_WINDOW_HZ)
                     & audit.local_minimum & (audit.baseline>0)]
    return float(eligible.ratio.min()) if len(eligible) else np.nan

for (model, mode, rep), curve in CURVES.items():
    if mode not in GENERATORS:
        continue
    P, S = curve['P'], curve['S']
    audit = points[(points.model==model) & (points['mode']==mode) & (points.rep==rep)].sort_values('f').reset_index(drop=True)
    for target in patch_nulls(P):
        if assign_site_family(target,P,S,CFG.site_assignment_tol_hz,CFG.site_ambiguity_hz)[0] != 'patch':
            continue
        ti = int(np.argmin(abs(audit.f.to_numpy()-target)))
        target_row = audit.iloc[ti]
        for offset in CONTROL_OFFSETS_HZ:
            row = dict(model=model, P=P, S=S, mode=mode, rep=rep,
                       target_hz=target, offset_hz=offset,
                       target_sample_hz=target_row.f, target_sample_error_hz=abs(target_row.f-target),
                       valid_pair=True, exclusion_reason='')
            reasons = []
            if abs(target_row.f-target)>TARGET_SAMPLE_TOL_HZ:
                reasons.append('target_not_sampled_exactly')
            for label, requested, tolerance in [('target',target,TARGET_SAMPLE_TOL_HZ),
                                                  ('left',target-offset,CONTROL_SAMPLE_TOL_HZ),
                                                  ('right',target+offset,CONTROL_SAMPLE_TOL_HZ)]:
                idx = int(np.argmin(abs(audit.f.to_numpy()-requested)))
                sample = audit.iloc[idx]
                row[label+'_requested_hz'] = requested
                row[label+'_sample_hz'] = sample.f
                row[label+'_sample_error_hz'] = abs(sample.f-requested)
                row[label+'_ratio'] = sample.ratio
                row[label+'_depth'] = 1-sample.ratio
                row[label+'_window_min_ratio'] = window_score(audit, sample.f)
                if abs(sample.f-requested)>tolerance:
                    reasons.append(label+'_sample_too_far')
                if not np.isfinite(sample.ratio):
                    reasons.append(label+'_invalid_baseline')
                window = audit[abs(audit.f-sample.f)<=CONTROL_WINDOW_HZ]
                row[label+'_window_points'] = len(window)
                if sample.f-CONTROL_WINDOW_HZ<BAND[0] or sample.f+CONTROL_WINDOW_HZ>BAND[1]:
                    reasons.append(label+'_window_outside_band')
                if label != 'target' and len(window):
                    nearest = float(grid_distance(window.f.to_numpy(),P,S).min())
                    row[label+'_min_grid_distance_hz'] = nearest
                    if nearest<=CFG.site_assignment_tol_hz:
                        reasons.append(label+'_window_near_theoretical_grid')
            row['valid_pair'] = not reasons
            row['exclusion_reason'] = ';'.join(dict.fromkeys(reasons))
            row['control_depth_mean'] = (row['left_depth']+row['right_depth'])/2
            row['paired_depth_delta'] = row['target_depth']-row['control_depth_mean']
            paired_rows.append(row)
paired_columns = ['model','P','S','mode','rep','target_hz','offset_hz','target_sample_hz',
    'target_sample_error_hz','valid_pair','exclusion_reason']
for label in ('target','left','right'):
    paired_columns += [label+s for s in ('_requested_hz','_sample_hz','_sample_error_hz','_ratio',
                                        '_depth','_window_min_ratio','_window_points')]
paired_columns += ['left_min_grid_distance_hz','right_min_grid_distance_hz','control_depth_mean','paired_depth_delta']
paired_audit = pd.DataFrame(paired_rows, columns=list(dict.fromkeys(paired_columns)))
valid_pairs = paired_audit[paired_audit.valid_pair.astype(bool)].copy()
pair_coverage = paired_audit.groupby(['mode','rep','offset_hz']).agg(
    candidate_pairs=('target_hz','size'), usable_pairs=('valid_pair','sum')).reset_index()
pair_coverage['excluded_pairs'] = pair_coverage.candidate_pairs-pair_coverage.usable_pairs
pair_exclusions = paired_audit[~paired_audit.valid_pair.astype(bool)].groupby(
    ['mode','offset_hz','exclusion_reason']).size().rename('rows_all_replicas').reset_index()
TABLES.update(paired_control_audit=paired_audit, paired_control_coverage=pair_coverage,
              paired_control_exclusions=pair_exclusions)
PROVENANCE['paired_controls'] = dict(offsets_hz=CONTROL_OFFSETS_HZ,
    control_sample_tolerance_hz=CONTROL_SAMPLE_TOL_HZ, target_sample_tolerance_hz=TARGET_SAMPLE_TOL_HZ,
    window_hz=CONTROL_WINDOW_HZ, grid_exclusion_hz=CFG.site_assignment_tol_hz,
    selection_uses_response=False, primary_offset_hz=4.0)
display(pair_coverage[pair_coverage.rep==-1])
display(pair_exclusions)
if valid_pairs.empty:
    warning('Nessuna coppia di controlli ammissibile: confronto continuo e quote non stimabili con questi criteri.')

paired_geometry = valid_pairs.groupby(['model','mode','rep','offset_hz']).agg(
    pairs=('target_hz','size'), target_depth_median=('target_depth','median'),
    control_depth_median=('control_depth_mean','median'),
    delta_median=('paired_depth_delta','median'),
    fraction_positive=('paired_depth_delta',lambda s: float((s>0).mean()))).reset_index()
paired_summary = valid_pairs.groupby(['mode','rep','offset_hz']).agg(
    pairs=('target_hz','size'), geometries=('model','nunique'),
    delta_median=('paired_depth_delta','median'),
    fraction_positive=('paired_depth_delta',lambda s: float((s>0).mean()))).reset_index()
TABLES.update(paired_depth_by_geometry=paired_geometry, paired_depth_summary=paired_summary)
display(paired_summary[paired_summary.rep==-1])
print('fraction_positive è una quota descrittiva di coppie; non è una probabilità posteriore.')

## 14. Stabilità del confronto e rilevazioni ai controlli

La curva media e le repliche sono riportate separatamente. La tabella di stabilità usa solo
candidati con coppie ammissibili in tutte le repliche della relativa geometria/generatore;
la quota di delta positivi misura coerenza descrittiva, non certezza statistica.
Le quote alle soglie alternative usano sempre lo stesso insieme di coppie ammissibili,
includendo come non rilevate le finestre che non hanno alcun minimo locale.

La griglia è non uniforme: finestre della stessa ampiezza possono contenere numeri diversi di campioni. La tabella delle quote riporta anche questa differenza; il confronto primario continuo usa un solo punto per candidato e per controllo.


In [ ]:
stability_rows = []
for key, g in valid_pairs[valid_pairs.rep>=0].groupby(['model','mode','target_hz','offset_hz']):
    model, mode, target, offset = key
    expected_reps = collapse[(collapse.model==model) & (collapse['mode']==mode)].rep.nunique()
    stability_rows.append(dict(model=model, mode=mode, target_hz=target, offset_hz=offset,
        available_replicas=len(g), expected_replicas=expected_reps,
        complete_replica_set=len(g)==expected_reps,
        positive_replicas=int((g.paired_depth_delta>0).sum()),
        median_delta=float(g.paired_depth_delta.median()),
        min_delta=float(g.paired_depth_delta.min()), max_delta=float(g.paired_depth_delta.max())))
pair_stability = pd.DataFrame(stability_rows, columns=['model','mode','target_hz','offset_hz',
    'available_replicas','expected_replicas','complete_replica_set','positive_replicas',
    'median_delta','min_delta','max_delta'])
rate_rows = []
for key, g in valid_pairs.groupby(['mode','rep','offset_hz']):
    mode, rep, offset = key
    for threshold in THRESHOLD_GRID:
        t = g.target_window_min_ratio.le(threshold).to_numpy(bool)
        l = g.left_window_min_ratio.le(threshold).to_numpy(bool)
        r = g.right_window_min_ratio.le(threshold).to_numpy(bool)
        control_rate = float((l.sum()+r.sum())/(2*len(g)))
        rate_rows.append(dict(mode=mode, rep=rep, offset_hz=offset, threshold=threshold,
            pairs=len(g), target_windows=len(g), control_windows=2*len(g),
            target_window_points_mean=float(g.target_window_points.mean()),
            control_window_points_mean=float((g.left_window_points.mean()+g.right_window_points.mean())/2),
            detected_target_windows=int(t.sum()), detected_control_windows=int(l.sum()+r.sum()),
            target_rate=float(t.mean()), control_rate=control_rate,
            rate_difference=float(t.mean())-control_rate))
window_rates = pd.DataFrame(rate_rows, columns=['mode','rep','offset_hz','threshold','pairs',
    'target_windows','control_windows','target_window_points_mean','control_window_points_mean','detected_target_windows','detected_control_windows',
    'target_rate','control_rate','rate_difference'])
TABLES.update(paired_depth_stability=pair_stability, paired_window_detection_rates=window_rates)
display(pair_stability.head(25))
display(window_rates.loc[(window_rates.rep==-1) & window_rates.threshold.isin([.75,.80,.85,.90]),
    ['mode','offset_hz','threshold','pairs','target_rate','control_rate','rate_difference',
     'target_window_points_mean','control_window_points_mean']])

fig, axes = plt.subplots(1, 2, figsize=(11,4), layout='constrained')
for column, color in [('patch','#2171b5'),('stride','#6a51a3'),('none','#d95f02')]:
    style = {'patch': dict(marker='o', ms=9, mfc='none'),
             'stride': dict(marker='s', ms=5), 'none': dict(marker='x', ms=7, ls='--')}[column]
    axes[0].plot(threshold_totals.threshold, threshold_totals[column], label=column, color=color, **style)
axes[0].axvline(.75,color='0.4',ls='--')
axes[0].axhline(10,color='0.5',ls=':',label='Barra di conteggio: 10')
axes[0].set(xlabel='Soglia del rapporto locale',ylabel='Minimi sulle curve medie generate',
            title='Sensibilità della selezione completa')
axes[0].legend()
for mode in GENERATORS:
    g = window_rates[(window_rates.rep==-1) & (window_rates.offset_hz==4.) & (window_rates['mode']==mode)]
    if len(g):
        line, = axes[1].plot(g.threshold,g.target_rate,marker='o',label=mode+' patch')
        axes[1].plot(g.threshold,g.control_rate,marker='x',ls='--',color=line.get_color(),label=mode+' controlli')
axes[1].axvline(.75,color='0.4',ls='--')
axes[1].set(xlabel='Soglia del rapporto locale',ylabel='Quota di finestre con un minimo rilevato',
            ylim=(-.02,1.02), title='Finestre appaiate, controlli a ±4 Hz')
handles, labels = axes[1].get_legend_handles_labels()
if handles: axes[1].legend(fontsize=8)
else: axes[1].text(.5,.5,'Nessuna coppia ammissibile',transform=axes[1].transAxes,ha='center')
FIGURES.append(('D2_threshold_sensitivity',fig))
plt.show()

fig, axes = plt.subplots(1,2,figsize=(11,4),layout='constrained',sharey=True)
for ax, offset in zip(axes, CONTROL_OFFSETS_HZ):
    for mode in GENERATORS:
        g = paired_geometry[(paired_geometry.rep==-1) & (paired_geometry.offset_hz==offset)
                             & (paired_geometry['mode']==mode)]
        if len(g):
            ax.scatter(g.control_depth_median,g.target_depth_median,label=mode,s=45,alpha=.75)
    limits = [*ax.get_xlim(), *ax.get_ylim()]
    low, high = min(limits), max(limits)
    ax.plot([low,high],[low,high],ls='--',color='0.5')
    ax.set(xlabel='Profondità mediana ai controlli',ylabel='Profondità mediana ai candidati patch',
           title=f'Una osservazione grafica per geometria/generatore, ±{offset:g} Hz')
    if ax.get_legend_handles_labels()[0]: ax.legend()
    else: ax.text(.5,.5,'Nessuna coppia ammissibile',transform=ax.transAxes,ha='center')
FIGURES.append(('D2_paired_continuous_depth',fig))
plt.show()

assert D2_IDENTIFICATION == BASELINE_D2_GATE
assert hashlib.sha256(pd.util.hash_pandas_object(sites, index=True).values.tobytes()).hexdigest() == BASELINE_SITES_HASH
print('D2 originale: gate, dati e modello invariati. Nessuna soglia alternativa selezionata.')

## 15. Audit mirato a 0.90: quali siti si recuperano?

La sensibilità FULL precedente ha trovato **22 siti patch sulle curve medie**, 10 KernelSynth e
12 TSMixup, a 0.90. È un riferimento storico da riprodurre, non un conteggio imposto. La soglia
0.90 richiede una depressione almeno del 10% rispetto alla mediana locale.

Si riutilizzano le rilevazioni già calcolate in `threshold_sites` per questa soglia; nessuna nuova
inferenza Chronos o MCMC. Si ricostruiscono le stesse statistiche del modello originale in tabelle
**alternative**, mantenendo anche le righe vuote. La frequenza del primo sito rimane `f1` e la
mediana delle differenze rimane `delta_hat`: non si dividono le risposte per l'indice armonico e
non si eliminano righe per farle coincidere con la previsione.

Per ogni sito patch medio si mostrano frequenza, armonica più vicina, novità rispetto a 0.75 e
repliche che rilevano un sito patch entro la tolleranza originale di 1.5 Hz. La stabilità è
descrittiva e non incrementa il conteggio dei siti medi.

In [ ]:
AUDIT_THRESHOLD = .90
audit090_events = threshold_sites[np.isclose(threshold_sites.threshold,AUDIT_THRESHOLD)].copy()
audit090_rows = []
for (model,mode,rep), curve in CURVES.items():
    if mode not in GENERATORS:
        continue
    events = audit090_events[(audit090_events.model==model) & (audit090_events['mode']==mode)
                             & (audit090_events.rep==rep)]
    for branch in ('stride','patch'):
        members = sorted(events.loc[events.family==branch,'frequency_hz'].to_list())
        delta = FS/(curve['S'] if branch=='stride' else curve['P'])
        stats = site_summary(members)
        first_harmonic = int(np.rint(stats['f1']/delta)) if members else np.nan
        first_error = abs(stats['f1']-first_harmonic*delta) if members else np.nan
        fundamental_present = bool(members and min(abs(np.asarray(members)-delta))<=CFG.site_assignment_tol_hz)
        audit090_rows.append(dict(model=model,P=curve['P'],S=curve['S'],mode=mode,rep=rep,
            branch=branch,predicted_spacing=delta,sites=' '.join(f'{f:.6f}' for f in members),
            first_harmonic=first_harmonic,first_harmonic_error_hz=first_error,
            fundamental_present=fundamental_present,
            fundamental_exclusive=assign_site_family(delta,curve['P'],curve['S'],
                CFG.site_assignment_tol_hz,CFG.site_ambiguity_hz)[0]==branch,
            f1_over_spacing=stats['f1']/delta if members else np.nan,
            delta_hat_over_spacing=stats['delta_hat']/delta if pd.notna(stats['delta_hat']) else np.nan,
            **stats))
audit090_sites = pd.DataFrame(audit090_rows,columns=['model','P','S','mode','rep','branch',
    'predicted_spacing','sites','first_harmonic','first_harmonic_error_hz',
    'fundamental_present','fundamental_exclusive','f1_over_spacing','delta_hat_over_spacing',
    'n_sites','f1','delta_hat'])
audit090_counts = audit090_sites[audit090_sites.rep==-1].groupby(['mode','branch']).n_sites.sum().unstack('branch',fill_value=0)
for branch in ('stride','patch'):
    actual = audit090_sites.loc[(audit090_sites.rep==-1) & audit090_sites.branch.eq(branch),'n_sites'].sum()
    previous = threshold_totals.loc[np.isclose(threshold_totals.threshold,AUDIT_THRESHOLD),branch].iloc[0]
    assert int(actual)==int(previous), 'Conteggi 0.90 non riconciliati con la sensibilità.'

audit090_support_rows = []
patch_mean_events = audit090_events[(audit090_events.rep==-1) & audit090_events.family.eq('patch')]
for event in patch_mean_events.itertuples():
    curve = CURVES[(event.model,event.mode,-1)]
    delta = FS/curve['P']
    replicas = sorted(rep for model,mode,rep in CURVES if model==event.model and mode==event.mode and rep>=0)
    hits, frequencies = [], []
    for rep in replicas:
        sub = audit090_events[(audit090_events.model==event.model) & (audit090_events['mode']==event.mode)
                              & (audit090_events.rep==rep) & audit090_events.family.eq('patch')]
        close = sub[abs(sub.frequency_hz-event.frequency_hz)<=CFG.site_assignment_tol_hz]
        if len(close):
            closest = close.iloc[np.argmin(abs(close.frequency_hz.to_numpy()-event.frequency_hz))]
            hits.append(int(rep));frequencies.append(float(closest.frequency_hz))
    old = merged_sites[(merged_sites.model==event.model) & (merged_sites['mode']==event.mode)
                       & (merged_sites.rep==-1) & merged_sites.family.eq('patch')]
    harmonic = int(np.rint(event.frequency_hz/delta))
    audit090_support_rows.append(dict(model=event.model,P=curve['P'],S=curve['S'],mode=event.mode,
        site_hz=event.frequency_hz,harmonic=harmonic,
        harmonic_error_hz=abs(event.frequency_hz-harmonic*delta),
        is_fundamental=bool(abs(event.frequency_hz-delta)<=CFG.site_assignment_tol_hz),
        new_vs075=not bool((abs(old.f-event.frequency_hz)<=CFG.site_assignment_tol_hz).any()),
        supported_replicas=len(hits),n_replicas=len(replicas),
        support_fraction=len(hits)/len(replicas),support_in_all_replicas=len(hits)==len(replicas),
        replicas=' '.join(map(str,hits)),replica_frequencies=' '.join(f'{f:.3f}' for f in frequencies),
        frequency_min_hz=min(frequencies) if frequencies else np.nan,
        frequency_max_hz=max(frequencies) if frequencies else np.nan))
audit090_support = pd.DataFrame(audit090_support_rows,columns=['model','P','S','mode','site_hz','harmonic',
    'harmonic_error_hz','is_fundamental','new_vs075','supported_replicas','n_replicas','support_fraction',
    'support_in_all_replicas','replicas','replica_frequencies','frequency_min_hz','frequency_max_hz'])
TABLES.update(audit090_sites=audit090_sites,audit090_patch_support=audit090_support,
              audit090_counts=audit090_counts.reset_index())
display(audit090_counts)
print('Elenco completo dei siti patch medi a 0.90, senza troncamento:')
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.width',160):
    display(audit090_support[['model','mode','site_hz','harmonic','new_vs075',
                             'supported_replicas','n_replicas','replica_frequencies']])
print('Il numero di siti è distinto dal numero di geometrie e dal numero di repliche.')

## 16. Compatibilità delle risposte con il D2 attuale

Il D2 primario confronta `f1` con $f_s/P$ o $f_s/S$. Un primo sito di armonica 2, 3, ... non è
automaticamente una misura della fondamentale: il modello può essere costruito numericamente,
ma la sua risposta non rappresenterebbe la quantità dichiarata. La tabella registra tutte le
righe non vuote, anche quelle incompatibili con questa interpretazione; non le filtra.

Per ciascuna geometria/generatore si controllano l'armonica del primo sito nella curva media,
la sua variabilità tra repliche, quante repliche rilevano la fondamentale e se questa è teoricamente
esclusiva del ramo. Si mostra anche quali serie a stride fisso coprono più valori di P: il semplice
totale di almeno dieci siti non basta a dimostrare movimento tra geometrie.

Per `delta_hat`, ogni intervallo tra siti viene spiegato attraverso il salto armonico. Le armoniche
intermedie sono distinte tra condivise (escluse dal metodo) ed esclusive non rilevate. Questa è una
diagnosi delle distanze, non una nuova risposta corretta da dare a D2.

In [ ]:
audit090_f1_inputs = audit090_sites[(audit090_sites.rep>=0) & (audit090_sites.n_sites>=1)].copy()
audit090_gap_inputs = audit090_sites[(audit090_sites.rep>=0) & (audit090_sites.n_sites>=2)
                                    & audit090_sites.delta_hat.notna()].copy()
endpoint_rows = []
for (model,mode,branch), g in audit090_sites.groupby(['model','mode','branch']):
    mean = g[g.rep==-1].iloc[0]
    reps = g[g.rep>=0]
    live = reps[reps.n_sites>=1]
    endpoint_rows.append(dict(model=model,mode=mode,branch=branch,P=int(mean.P),S=int(mean.S),
        mean_n_sites=int(mean.n_sites),mean_f1=mean.f1,mean_first_harmonic=mean.first_harmonic,
        mean_fundamental_present=bool(mean.fundamental_present),
        fundamental_exclusive=bool(mean.fundamental_exclusive),
        n_replicas=len(reps),replicas_with_any_site=len(live),
        replicas_with_fundamental=int(reps.fundamental_present.sum()),
        distinct_first_harmonics=int(live.first_harmonic.nunique()),
        replica_f1_min=float(live.f1.min()) if len(live) else np.nan,
        replica_f1_max=float(live.f1.max()) if len(live) else np.nan,
        replica_f1_span_hz=float(live.f1.max()-live.f1.min()) if len(live) else np.nan))
audit090_endpoints = pd.DataFrame(endpoint_rows)
input_rows=[]
for branch in ('stride','patch'):
    sub=audit090_f1_inputs[audit090_f1_inputs.branch==branch]
    gap_sub=audit090_gap_inputs[audit090_gap_inputs.branch==branch]
    mean=audit090_sites[(audit090_sites.rep==-1) & audit090_sites.branch.eq(branch)]
    input_rows.append(dict(branch=branch,mean_sites=int(mean.n_sites.sum()),
        passes_site_count_only=bool(mean.n_sites.sum()>=CFG.min_d2_sites),
        f1_rows=len(sub),geometries=int(sub.model.nunique()),predictor_values=int(sub.predicted_spacing.nunique()),
        f1_rows_with_fundamental=int(sub.fundamental_present.sum()),
        f1_rows_without_fundamental=int((~sub.fundamental_present).sum()),
        gap_rows=len(gap_sub),
        gap_rows_different_from_spacing=int((abs(gap_sub.delta_hat-gap_sub.predicted_spacing)>CFG.collapse_step).sum())))
audit090_input_summary=pd.DataFrame(input_rows)

series_rows=[]
patch=audit090_f1_inputs[audit090_f1_inputs.branch=='patch']
for S,g in patch.groupby('S'):
    first=g[g.fundamental_present]
    series_rows.append(dict(S=int(S),patch_sizes=' '.join(map(str,sorted(g.P.unique()))),
        n_patch_sizes=int(g.P.nunique()),rows=len(g),
        patch_sizes_with_fundamental=' '.join(map(str,sorted(first.P.unique()))),
        n_patch_sizes_with_fundamental=int(first.P.nunique()),
        rows_with_fundamental=len(first)))
audit090_patch_series=pd.DataFrame(series_rows,columns=['S','patch_sizes','n_patch_sizes','rows',
    'patch_sizes_with_fundamental','n_patch_sizes_with_fundamental','rows_with_fundamental'])

gap_rows=[]
for row in audit090_sites[audit090_sites.n_sites>=2].itertuples():
    curve_events=audit090_events[(audit090_events.model==row.model) & (audit090_events['mode']==row.mode)
                                 & (audit090_events.rep==row.rep) & audit090_events.family.eq(row.branch)]
    frequencies=np.sort(curve_events.frequency_hz.to_numpy(float))
    delta=row.predicted_spacing
    harmonic=np.rint(frequencies/delta).astype(int)
    for j in range(len(frequencies)-1):
        jump=int(harmonic[j+1]-harmonic[j])
        skipped=list(range(int(harmonic[j])+1,int(harmonic[j+1])))
        skipped_families=[assign_site_family(h*delta,row.P,row.S,CFG.site_assignment_tol_hz,
                                            CFG.site_ambiguity_hz)[0] for h in skipped]
        gap_rows.append(dict(model=row.model,mode=row.mode,rep=row.rep,branch=row.branch,
            left_hz=frequencies[j],right_hz=frequencies[j+1],
            left_harmonic=int(harmonic[j]),right_harmonic=int(harmonic[j+1]),harmonic_jump=jump,
            gap_hz=frequencies[j+1]-frequencies[j],predicted_spacing=delta,
            gap_residual_from_harmonics_hz=(frequencies[j+1]-frequencies[j])-jump*delta,
            shared_intermediate_harmonics=skipped_families.count('both'),
            exclusive_missing_harmonics=skipped_families.count(row.branch),
            other_intermediate_harmonics=len(skipped_families)-skipped_families.count('both')-skipped_families.count(row.branch)))
audit090_gaps=pd.DataFrame(gap_rows,columns=['model','mode','rep','branch','left_hz','right_hz',
    'left_harmonic','right_harmonic','harmonic_jump','gap_hz','predicted_spacing',
    'gap_residual_from_harmonics_hz','shared_intermediate_harmonics','exclusive_missing_harmonics',
    'other_intermediate_harmonics'])
TABLES.update(audit090_f1_inputs=audit090_f1_inputs,audit090_gap_inputs=audit090_gap_inputs,
    audit090_endpoints=audit090_endpoints,audit090_input_summary=audit090_input_summary,
    audit090_patch_series=audit090_patch_series,audit090_gaps=audit090_gaps)
display(audit090_input_summary)
print('Risposta primaria patch per geometria/generatore, inclusi i siti presenti solo nelle repliche:')
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.width',160):
    display(audit090_endpoints[(audit090_endpoints.branch=='patch')
        & ((audit090_endpoints.mean_n_sites>0) | (audit090_endpoints.replicas_with_any_site>0))][
        ['model','mode','mean_f1','mean_first_harmonic','n_replicas','replicas_with_any_site',
         'replicas_with_fundamental','distinct_first_harmonics','replica_f1_span_hz']])
display(audit090_patch_series)
print('Intervalli non consecutivi delle curve medie, elenco completo:')
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.width',160):
    display(audit090_gaps[(audit090_gaps.rep==-1) & (audit090_gaps.harmonic_jump!=1)][
        ['model','mode','branch','left_hz','right_hz','harmonic_jump','gap_hz',
         'shared_intermediate_harmonics','exclusive_missing_harmonics']])

## 17. Riscontro finale su 0.90, senza scegliere una nuova analisi

Il grafico mostra quante repliche sostengono ciascun sito patch della curva media. Un sito
confermato in tutte le repliche può comunque essere un'armonica successiva; stabilità e validità
di `f1` sono controlli distinti. La tabella conclusiva riporta le criticità senza eliminare siti o
decretare automaticamente che una soglia sia accettabile.

Gli elenchi completi sono stampati nel notebook e salvati nelle tabelle `audit090_*`, così il
notebook eseguito contiene i dettagli necessari per discuterli insieme. Se anche tutti questi
controlli risultassero favorevoli, la scelta di 0.90 resterebbe esplorativa sugli stessi dati.

In [ ]:
audit090_support_summary=audit090_support.groupby('mode').agg(
    sites=('site_hz','size'),new_vs075=('new_vs075','sum'),
    supported_in_all_replicas=('support_in_all_replicas','sum'),
    median_support_fraction=('support_fraction','median'),
    sites_at_fundamental=('is_fundamental','sum')).reset_index()
TABLES['audit090_support_summary']=audit090_support_summary
display(audit090_support_summary)

if len(audit090_support):
    chart=audit090_support.sort_values(['model','mode','site_hz']).reset_index(drop=True)
    fig,ax=plt.subplots(figsize=(11,max(4,.30*len(chart)+1.5)),layout='constrained')
    positions=np.arange(len(chart))
    labels=[f'{r.model} / {r.mode} / {r.site_hz:.2f} Hz / h={r.harmonic}' for r in chart.itertuples()]
    ax.barh(positions,chart.n_replicas,color='0.9',label='Repliche disponibili')
    ax.barh(positions,chart.supported_replicas,color=[COLORS['patch'] if n else '#6a51a3' for n in chart.new_vs075],
            label='Repliche con rilevazione patch')
    ax.set_yticks(positions,labels);ax.invert_yaxis()
    ax.set_xticks(np.arange(int(chart.n_replicas.max())+1))
    ax.set(xlabel='Numero di repliche',title='Siti patch medi a 0.90: stabilità e armonica')
    ax.legend(loc='lower right',fontsize=8)
    FIGURES.append(('D2_090_patch_replica_support',fig));plt.show()
else:
    print('Nessun sito patch medio a 0.90; grafico di stabilità non applicabile.')

AUDIT090_NOTES=[]
patch_info=audit090_input_summary[audit090_input_summary.branch=='patch'].iloc[0]
if not patch_info.passes_site_count_only:
    AUDIT090_NOTES.append('Anche a 0.90 il ramo patch non supera il conteggio minimo di dieci siti medi.')
if patch_info.f1_rows_without_fundamental:
    AUDIT090_NOTES.append(f"A 0.90, {int(patch_info.f1_rows_without_fundamental)}/{int(patch_info.f1_rows)} righe patch non vuote delle repliche non rilevano la fondamentale: il loro f1 non va interpretato automaticamente come fs/P.")
if len(audit090_support):
    partial=int((~audit090_support.support_in_all_replicas).sum())
    AUDIT090_NOTES.append(f'{partial}/{len(audit090_support)} siti patch medi non sono confermati in tutte le repliche disponibili.')
if audit090_patch_series.empty or not (audit090_patch_series.n_patch_sizes>=2).any():
    AUDIT090_NOTES.append('Le righe patch disponibili non coprono più valori di P a stride fisso.')
if audit090_patch_series.empty or not (audit090_patch_series.n_patch_sizes_with_fundamental>=2).any():
    AUDIT090_NOTES.append('Non risulta una serie a stride fisso con la fondamentale rilevata in almeno due valori di P; questa descrizione non seleziona nuove righe per il fit.')
shared_gaps=int(((audit090_gaps.rep==-1) & (audit090_gaps.shared_intermediate_harmonics>0)).sum())
missing_gaps=int(((audit090_gaps.rep==-1) & (audit090_gaps.exclusive_missing_harmonics>0)).sum())
AUDIT090_NOTES.append(f'Intervalli sulle curve medie generate, entrambi i rami: {shared_gaps} attraversano armoniche condivise escluse; {missing_gaps} attraversano armoniche esclusive non rilevate. Le due categorie possono sovrapporsi.')
for message in AUDIT090_NOTES:print(message)
assert D2_IDENTIFICATION==BASELINE_D2_GATE
assert hashlib.sha256(pd.util.hash_pandas_object(sites,index=True).values.tobytes()).hexdigest()==BASELINE_SITES_HASH
PROVENANCE['audit090']=dict(threshold=AUDIT_THRESHOLD,exploratory=True,no_refit=True,
    original_D2_gate_unchanged=True,notes=AUDIT090_NOTES)
print('Audit 0.90 completato: nessun refit, nessuna correzione automatica delle risposte.')

## 18. Riepilogo e file da discutere insieme

Si separano conteggi osservati, riproducibilità della selezione e possibili spiegazioni.
Il riepilogo non promuove i controlli diagnostici a supporto o confutazione di H3.
L'assenza di minimi rilevati può dipendere dall'indicatore, dal rilevatore o dai dati;
le tabelle aiutano a decidere il prossimo passo senza scegliere ora una correzione.


In [ ]:
root = Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT else DATA_DIR.parent / 'd2_diagnostics'
if root == DATA_DIR or DATA_DIR in root.parents:
    raise ValueError('OUTPUT_ROOT deve essere esterna alla cartella dei Parquet originali.')
OUTPUT_DIR = root / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
for name, table in TABLES.items():
    table.to_parquet(OUTPUT_DIR / f'{name}.parquet', index=False, engine=PARQUET_ENGINE)
    if name not in ('point_audit','sites_reconciliation'):
        table.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
for name, fig in FIGURES:
    fig.savefig(OUTPUT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

summary_lines = [
    '# Diagnosi D2, modello e soglie invariati',
    '',
    f'Sorgente: {DATA_DIR}',
    f'Siti salvati sulle curve medie generate: stride={int(totals["stride"])}, patch={int(totals["patch"])}.',
    f'Gate attuale (10): {D2_IDENTIFICATION}.',
    f'Riproduzione dei siti dai collapse: {REPLAY_MATCHES} ({int(comparison.matches.sum())}/{len(comparison)} righe).',
    f'Provenienza verificata con manifest: {manifest_verified}.',
    f'Factory D2: {MODEL_GRAPH_STATUS}; nessuna nuova posteriore.',
    '',
    '## Riscontri da esaminare',
]
for row in patch_saved.itertuples():
    summary_lines.append(f'- Patch {row.model}/{row.mode}: {row.n_sites} siti, frequenze {row.sites} Hz.')
for row in support[support.family.eq('patch')].itertuples():
    summary_lines.append(f'- {row.model}/{row.mode}, {row.mean_site_hz:.3f} Hz: '
                         f'{row.supported_replicas}/{row.n_replicas} repliche confermano il ramo patch entro 1,5 Hz.')
for row in candidate_summary.itertuples():
    summary_lines.append(f'- Candidati patch esclusivi, {row.mode}: {row.reason} = {row.candidate_count}.')
bad_fundamental = harmonics[(harmonics.rep==-1) & harmonics['mode'].isin(GENERATORS)
                            & ~harmonics.fundamental_detected.astype(bool)]
summary_lines += [f"- Righe medie generate in cui il primo sito non coincide con l'armonica 1: {len(bad_fundamental)}.",
    '', '## Interpretazione e prossimo passo',
    '- Se il replay non coincide, risolvere prima la provenienza/versione del rilevatore.',
    '- Leggere patch_candidates e i grafici: distinguere assenza di minimo, profondità insufficiente, fusione e attribuzione both.',
    '- Confrontare pure e background generati; il test delle medie di patch è solo geometrico, non un test degli embedding.',
    '- Leggere harmonic_audit e gap_audit: armoniche mancanti o condivise possono cambiare f1 e delta_hat.',
    '- Nessuna soglia abbassata, nessun sito aggiunto e nessun dato trasformato per far superare il gate.',
    '', '## Limiti di questa esecuzione']

summary_lines += ['- ' + w for w in WARNINGS] or ['- Nessuna anomalia di copertura/provenienza segnalata; questo non è un verdetto bayesiano.']
summary_lines += ['', '## Sensibilità esplorativa e controlli appaiati',
    '- Il riferimento resta 0.75. Non è stata scelta una soglia alternativa e D2 non è stato rifittato.',
    '- Confronti sugli stessi dati, senza validazione indipendente; quote ai controlli non equivalgono a falsi positivi certificati.']
for row in threshold_totals.itertuples():
    summary_lines.append(f'- Soglia {row.threshold:.2f}: patch={row.patch}, stride={row.stride}, fuori griglia={row.none}; solo conteggi alternativi.')
for row in paired_summary[paired_summary.rep==-1].itertuples():
    summary_lines.append(f'- {row.mode}, offset {row.offset_hz:g} Hz: {row.pairs} coppie in {row.geometries} geometrie; '
                         f'delta mediano={row.delta_median:.4f}, quota delta positivi={row.fraction_positive:.3f}.')
for row in window_rates[(window_rates.rep==-1) & window_rates.threshold.isin([.75,.80,.85,.90])].itertuples():
    summary_lines.append(f'- {row.mode}, offset {row.offset_hz:g}, soglia {row.threshold:.2f}: '
                         f'finestre patch rilevate={row.target_rate:.3f}, controlli={row.control_rate:.3f}; {row.pairs} coppie.')
summary_lines += ['- Prima di adottare una soglia: esaminare copertura dei controlli, differenza patch-controlli, stabilità tra repliche e comportamento a entrambi gli offset.',
    '- Una soglia non risolve da sola armoniche mancanti o f1 diverso dalla fondamentale.']



summary_lines += ['', '## Audit mirato a 0.90']
for row in audit090_support_summary.itertuples():
    summary_lines.append(f'- {row.mode}: {row.sites} siti patch medi, {row.new_vs075} nuovi rispetto a 0.75; {row.supported_in_all_replicas} confermati in tutte le repliche; {row.sites_at_fundamental} alla fondamentale.')
summary_lines += ['- '+message for message in AUDIT090_NOTES]
summary_lines += ['- I dettagli completi sono nelle tabelle audit090_*; D2 e il gate originale restano invariati.']

for name, path in INPUTS.items():
    if sha256(path) != INPUT_HASHES[name]:
        raise RuntimeError('Un Parquet sorgente è cambiato durante la diagnosi. Non usare questi risultati.')
if manifest and sha256(manifest_path) != PROVENANCE['manifest_sha256']:
    raise RuntimeError('Il manifest sorgente è cambiato durante la diagnosi.')
PROVENANCE.update(warnings=WARNINGS, replay_matches=REPLAY_MATCHES,
                  gate=D2_IDENTIFICATION, output_dir=str(OUTPUT_DIR), no_sampling=True,
                  original_inputs_unchanged=True)
(OUTPUT_DIR / 'provenance.json').write_text(json.dumps(PROVENANCE, indent=2), encoding='utf-8')
summary_text = '\n'.join(summary_lines)
(OUTPUT_DIR / 'SUMMARY.md').write_text(summary_text, encoding='utf-8')
display(Markdown(summary_text))
print('Output diagnostici:', OUTPUT_DIR)
print('Condividere SUMMARY.md, provenance.json e i grafici per scegliere insieme il passo successivo.')
